In [ ]:
import numpy as np  # مكتبة العمليات الرياضية والمصفوفات
import matplotlib.pyplot as plt  # مكتبة الرسم البياني
import pandas as pd  # مكتبة معالجة البيانات (DataFrame)

from sklearn.model_selection import train_test_split, StratifiedKFold  # تقسيم البيانات + Cross Validation
from sklearn.preprocessing import LabelEncoder, OneHotEncoder  # تحويل البيانات النصية إلى رقمية
from sklearn.linear_model import LogisticRegression  # نموذج الانحدار اللوجستي للتصنيف
from sklearn.metrics import accuracy_score  # حساب دقة النموذج

In [ ]:
file = r"C:\Users\HP\Desktop\New folder (2)\titanic.xlsx"  # مسار ملف البيانات

titanic_data = pd.read_excel(file)  # قراءة ملف Excel وتحويله إلى DataFrame

print(titanic_data.head())  # عرض أول 5 صفوف من البيانات

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [ ]:
print(titanic_data.isnull().sum())  # حساب القيم المفقودة في كل عمود
print(titanic_data.shape)  # عرض عدد الصفوف والأعمدة

missing_data = titanic_data.isnull().sum()  # تخزين عدد القيم المفقودة لكل عمود
print(missing_data)  # طباعة القيم المفقودة

rate_missing = missing_data / titanic_data.shape[0]  # حساب نسبة القيم المفقودة
print(rate_missing)  # طباعة النسبة

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
(891, 12)
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
PassengerId    0.000000
Survived       0.000000
Pclass         0.000000
Name           0.000000
Sex            0.000000
Age            0.198653
SibSp          0.000000
Parch          0.000000
Ticket         0.000000
Fare           0.000000
Cabin          0.771044
Embarked       0.002245
dtype: float64


In [ ]:
train_data, test_data = train_test_split(  # تقسيم البيانات إلى تدريب واختبار
    titanic_data,
    test_size=0.3,  # 30% اختبار
    random_state=42  # تثبيت العشوائية
)

print(train_data.shape, test_data.shape)  # عرض حجم بيانات التدريب والاختبار

(623, 12) (268, 12)


In [ ]:
Strat_kfold = StratifiedKFold(  # إنشاء Cross Validation مع الحفاظ على توزيع الفئات
    n_splits=5,  # تقسيم البيانات إلى 5 أجزاء
    shuffle=True,  # خلط البيانات قبل التقسيم
    random_state=42  # تثبيت العشوائية
)

x = train_data.drop('Survived', axis='columns')  # حذف العمود الهدف من المدخلات
y = train_data['Survived']  # تحديد العمود الهدف (التوقع)

In [ ]:
scores = []  # قائمة لتخزين دقة كل Fold

one_hot_en = OneHotEncoder(  # تحويل القيم النصية إلى One-Hot Encoding
    sparse_output=False,  # إرجاع مصفوفة عادية بدل sparse
    handle_unknown='ignore'  # تجاهل القيم غير المعروفة
)

label_incoder = LabelEncoder()  # تحويل القيم النصية إلى أرقام

In [ ]:
for train_index, val_index in Strat_kfold.split(x, y):  # تنفيذ Cross Validation

    x_train_fold = x.iloc[train_index].copy()  # بيانات التدريب لكل Fold
    x_val_fold = x.iloc[val_index].copy()  # بيانات التحقق لكل Fold

    y_train_fold = y.iloc[train_index]  # الهدف للتدريب
    y_val_fold = y.iloc[val_index]  # الهدف للتحقق

    # Age
    median_age = x_train_fold['Age'].median()  # حساب الوسيط للعمر
    x_train_fold['Age'] = x_train_fold['Age'].fillna(median_age)  # تعويض القيم المفقودة في التدريب
    x_val_fold['Age'] = x_val_fold['Age'].fillna(median_age)  # تعويض القيم المفقودة في التحقق

    # Cabin
    x_train_fold['Cabin'] = x_train_fold['Cabin'].fillna('Unknown')  # تعويض القيم المفقودة في Cabin
    x_val_fold['Cabin'] = x_val_fold['Cabin'].fillna('Unknown')  # نفس الشيء في التحقق

    # Embarked
    most_frequent_embarked = x_train_fold['Embarked'].mode()[0]  # أكثر قيمة تكرارًا
    x_train_fold['Embarked'] = x_train_fold['Embarked'].fillna(most_frequent_embarked)  # تعويض التدريب
    x_val_fold['Embarked'] = x_val_fold['Embarked'].fillna(most_frequent_embarked)  # تعويض التحقق

    # Sex One Hot
    sex_encoded_train = one_hot_en.fit_transform(x_train_fold[['Sex']])  # تحويل Sex إلى One-Hot (تدريب)
    sex_encoded_val = one_hot_en.transform(x_val_fold[['Sex']])  # تطبيق نفس التحويل على التحقق

    # Mean Encoding
    temp_df = x_train_fold.copy()  # نسخة من بيانات التدريب
    temp_df['Survived'] = y_train_fold  # إضافة العمود الهدف

    embarked_mean_encoding = temp_df.groupby('Embarked')['Survived'].mean().to_dict()  # متوسط النجاة لكل فئة

    embarked_encoded_train = x_train_fold['Embarked'].map(embarked_mean_encoding)  # تحويل Embarked إلى قيم رقمية
    embarked_encoded_val = x_val_fold['Embarked'].map(embarked_mean_encoding)  # نفس التحويل للـ validation

    # Label Encoding Cabin
    label_incoder.fit(pd.concat([x_train_fold['Cabin'], x_val_fold['Cabin']]))  # تدريب Label Encoder

    cabin_encoded_train = label_incoder.transform(x_train_fold['Cabin'])  # تحويل Cabin إلى أرقام (تدريب)
    cabin_encoded_val = label_incoder.transform(x_val_fold['Cabin'])  # تحويل Cabin (تحقق)

    # تجهيز البيانات النهائية
    x_train_fold = x_train_fold[['Pclass','Age','SibSp','Parch','Fare']].copy()  # اختيار الأعمدة المهمة
    x_val_fold = x_val_fold[['Pclass','Age','SibSp','Parch','Fare']].copy()  # نفس الأعمدة للتحقق

    x_train_fold['Embarked_encoded'] = embarked_encoded_train  # إضافة Embarked المشفر
    x_val_fold['Embarked_encoded'] = embarked_encoded_val

    x_train_fold['Cabin_encoded'] = cabin_encoded_train  # إضافة Cabin المشفر
    x_val_fold['Cabin_encoded'] = cabin_encoded_val

    # Sex columns
    columns = ['Sex_encoded_' + str(i) for i in range(sex_encoded_train.shape[1])]  # أسماء أعمدة Sex

    for i, col in enumerate(columns):  # إضافة أعمدة Sex المشفرة
        x_train_fold[col] = sex_encoded_train[:, i]
        x_val_fold[col] = sex_encoded_val[:, i]

    # Model
    model = LogisticRegression(max_iter=1000)  # إنشاء نموذج Logistic Regression
    model.fit(x_train_fold, y_train_fold)  # تدريب النموذج

    y_pred = model.predict(x_val_fold)  # التنبؤ على بيانات التحقق

    accuracy = accuracy_score(y_val_fold, y_pred)  # حساب الدقة
    scores.append(accuracy)  # تخزين الدقة

In [ ]:
print("Fold Accuracies:", scores)  # طباعة دقة كل Fold
print("Average Accuracy:", np.mean(scores))  # طباعة متوسط الدقة

Fold Accuracies: [0.792, 0.872, 0.808, 0.7903225806451613, 0.7258064516129032]
Average Accuracy: 0.7976258064516129


In [ ]:
print(titanic_data.head())  # عرض أول 5 صفوف من البيانات



   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [ ]:
print(titanic_data.head(5))  # عرض أول 5 صفوف من البيانات



   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [ ]:
print(titanic_data.head(2))

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   

   Parch     Ticket     Fare Cabin Embarked  
0      0  A/5 21171   7.2500   NaN        S  
1      0   PC 17599  71.2833   C85        C  


In [ ]:
print(titanic_data.head(1))

   PassengerId  Survived  Pclass                     Name   Sex   Age  SibSp  \
0            1         0       3  Braund, Mr. Owen Harris  male  22.0      1   

   Parch     Ticket  Fare Cabin Embarked  
0      0  A/5 21171  7.25   NaN        S  


In [ ]:
print(titanic_data.shape)

(891, 12)


In [ ]:
print(rate_missing)  # طباعة النسبة

PassengerId    0.000000
Survived       0.000000
Pclass         0.000000
Name           0.000000
Sex            0.000000
Age            0.198653
SibSp          0.000000
Parch          0.000000
Ticket         0.000000
Fare           0.000000
Cabin          0.771044
Embarked       0.002245
dtype: float64


In [ ]:
rate_missing = missing_data / titanic_data.shape[0]  # حساب نسبة القيم المفقودة
print(rate_missing)

PassengerId    0.000000
Survived       0.000000
Pclass         0.000000
Name           0.000000
Sex            0.000000
Age            0.198653
SibSp          0.000000
Parch          0.000000
Ticket         0.000000
Fare           0.000000
Cabin          0.771044
Embarked       0.002245
dtype: float64


In [ ]:
print("Fold Accuracies:", scores)

Fold Accuracies: [0.792, 0.872, 0.808, 0.7903225806451613, 0.7258064516129032]


In [ ]:
accuracy = accuracy_score(y_val_fold, y_pred)  # حساب الدقة
    scores.append(accuracy)  # تخزين الدقة

IndentationError: unexpected indent (<ipython-input-17-4b6da89c153a>, line 2)

In [ ]:
accuracy = accuracy_score(y_val_fold, y_pred) 
    scores.append(accuracy)

IndentationError: unexpected indent (<ipython-input-18-cdbadfe5281b>, line 2)

In [ ]:
accuracy = accuracy_score(y_val_fold, y_pred) 
scores.append(accuracy)

In [ ]:
accuracy = accuracy_score(y_val_fold, y_pred) 
scores.append(accuracy)

In [ ]:
accuracy = accuracy_score(y_val_fold, y_pred)

In [ ]:
 accuracy = accuracy_score(y_val_fold, y_pred)  # حساب الدقة
    scores.append(accuracy)  # تخزين الدقة
    print("accuracy")

IndentationError: unexpected indent (<ipython-input-22-80860ea18cb8>, line 2)

In [ ]:
accuracy = accuracy_score(y_val_fold, y_pred)

scores.append(accuracy)

print(accuracy)


0.7258064516129032


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

# تقسيم البيانات
train_data, test_data = train_test_split(
    titanic_data,
    test_size=0.40,   # 40% للاختبار
    train_size=0.60,  # 60% للتدريب
    random_state=42
)

# إنشاء Cross Validation بعدد 5 تكرارات
Strat_kfold = StratifiedKFold(
    n_splits=5,   # عدد التكرارات (Folds)
    shuffle=True,
    random_state=42
)

In [ ]:
train_data, test_data = train_test_split(  # تقسيم البيانات إلى تدريب واختبار
    titanic_data,
    test_size=0.4,  # 30% اختبار
   train_size=0.6,
    random_state=42  # تثبيت العشوائية
)

print(train_data.shape, test_data.shape)  # عرض حجم بيانات التدريب والاختبار

(534, 12) (357, 12)


In [ ]:
columns = ['Sex_encoded_' + str(i) for i in range(sex_encoded_train.shape[1])]  # أسماء أعمدة Sex



In [ ]:
columns = ['Sex_encoded_' + str(i) for i in range(sex_encoded_train.shape[1])]  # أسماء أعمدة Sex
print(columns)


['Sex_encoded_0', 'Sex_encoded_1']


In [ ]:
for col in titanic_data.columns:
    
    print("\nاسم العمود:", col)

    print(titanic_data[col].value_counts())



اسم العمود: PassengerId
PassengerId
1      1
2      1
3      1
4      1
5      1
      ..
887    1
888    1
889    1
890    1
891    1
Name: count, Length: 891, dtype: int64

اسم العمود: Survived
Survived
0    549
1    342
Name: count, dtype: int64

اسم العمود: Pclass
Pclass
3    491
1    216
2    184
Name: count, dtype: int64

اسم العمود: Name
Name
Braund, Mr. Owen Harris                                1
Cumings, Mrs. John Bradley (Florence Briggs Thayer)    1
Heikkinen, Miss. Laina                                 1
Futrelle, Mrs. Jacques Heath (Lily May Peel)           1
Allen, Mr. William Henry                               1
                                                      ..
Montvila, Rev. Juozas                                  1
Graham, Miss. Margaret Edith                           1
Johnston, Miss. Catherine Helen "Carrie"               1
Behr, Mr. Karl Howell                                  1
Dooley, Mr. Patrick                                    1
Name: count, Length:

In [ ]:
rate_missing = missing_data / titanic_data.shape[0]  # حساب نسبة القيم المفقودة
print(rate_missing)  # طباعة النسبة

PassengerId    0.000000
Survived       0.000000
Pclass         0.000000
Name           0.000000
Sex            0.000000
Age            0.198653
SibSp          0.000000
Parch          0.000000
Ticket         0.000000
Fare           0.000000
Cabin          0.771044
Embarked       0.002245
dtype: float64


In [ ]:
print(titanic_data.shape)  

(891, 12)


In [ ]:
print(titanic_data.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


In [ ]:
titanic_data['c']

KeyError: 'c'

In [ ]:
titanic_data['Cabin'].isnull().sum()

np.int64(687)

In [ ]:
 x_train_fold['Cabin'] = x_train_fold['Cabin'].fillna('Unknown')


KeyError: 'Cabin'

In [ ]:
titanic_data['Cabin'].isnull().sum()
x_train_fold['Cabin'] = x_train_fold['Cabin'].fillna('Unknown')


KeyError: 'Cabin'

In [ ]:
print(titanic_data.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


In [ ]:
titanic_data['Cabin'] = titanic_data['Cabin'].fillna('Unknown')

In [ ]:
 x_train_fold['Age'] = x_train_fold['Age'].fillna(median_age)

In [ ]:
prin

NameError: name 'prin' is not defined

In [ ]:
print(age)

NameError: name 'age' is not defined

In [ ]:
print(titanic_data['Cabin'].head(10))

0    Unknown
1        C85
2    Unknown
3       C123
4    Unknown
5    Unknown
6        E46
7    Unknown
8    Unknown
9    Unknown
Name: Cabin, dtype: object


In [ ]:
print (titanic_data.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')
